# PDB 構造情報サマリー

複数の PDB ID について、RCSB Data API から構造情報と chain 情報を取得し、
RSCC / BSA 解析前の確認用テーブルを作成します。

**出力 CSV:**
- `pdb_summary.csv` — PDB ごとのタイトル・解像度・引用情報
- `pdb_chain_info.csv` — PDB × chain の entity 説明（long format）


In [ ]:
# Cell 1: 必要ライブラリの import
from __future__ import annotations

import logging
from pathlib import Path

import pandas as pd
import requests

# RCSB Data API のベース URL
RCSB_CORE_URL = "https://data.rcsb.org/rest/v1/core"
REQUEST_TIMEOUT = 30  # 秒

# ログ設定（取得失敗 PDB のスキップ理由を表示）
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

# Notebook / 出力ディレクトリ
NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "structure_tools" / "PDB_analysis").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "structure_tools" / "PDB_analysis"
OUTPUT_DIR = NOTEBOOK_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR.resolve()}")



In [ ]:
from __future__ import annotations

# Cell 2: 対象 PDB ID 一覧
# RSCC / BSA 解析前に確認したい SARS-CoV-2 RTC 関連構造などをここに列挙します。

pdb_ids = [
    "6XEZ",
    "7CXM",
    "7CXN",
    "7RDX",
    "7RDY",
    "7RDZ",
    "7RE0",
    "7RE1",
    "7RE2",
    "7RE3",
]

# 重複除去 & 大文字正規化
pdb_ids = sorted({pdb_id.strip().upper() for pdb_id in pdb_ids})
print(f"Target PDB count: {len(pdb_ids)}")
print(pdb_ids)



In [ ]:
from __future__ import annotations

# Cell 3: RCSB Data API から entry サマリーを取得
# 取得項目: title, resolution, citation (DOI / first author / journal / year)
# 取得に失敗した PDB はスキップし、ログに理由を出力します。


def fetch_json(url: str) -> dict:
    """RCSB REST API から JSON を取得する。"""
    response = requests.get(url, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    return response.json()


def parse_resolution(entry: dict) -> float | None:
    """entry から解像度 (Å) を取得する。"""
    res = entry.get("rcsb_entry_info", {}).get("resolution_combined")
    if res is None:
        return None
    if isinstance(res, list):
        if not res:
            return None
        return float(res[0])
    return float(res)


def parse_citation(entry: dict) -> dict:
    """primary citation から DOI / 著者 / ジャーナル / 年を取得する。"""
    cite = entry.get("rcsb_primary_citation") or {}
    authors = cite.get("rcsb_authors") or []
    first_author = authors[0] if authors else None
    doi = cite.get("pdbx_database_id_DOI") or cite.get("doi")
    journal = cite.get("journal_abbrev") or cite.get("rcsb_journal_abbrev")
    year = cite.get("year")
    return {
        "doi": doi,
        "first_author": first_author,
        "journal": journal,
        "year": year,
    }


def fetch_entry_summary(pdb_id: str) -> dict:
    """1 PDB 分の summary 行を dict で返す。"""
    url = f"{RCSB_CORE_URL}/entry/{pdb_id.upper()}"
    entry = fetch_json(url)
    citation = parse_citation(entry)
    resolution = parse_resolution(entry)

    return {
        "pdb_id": pdb_id.upper(),
        "title": entry.get("struct", {}).get("title"),
        "resolution_A": resolution,
        "doi": citation["doi"],
        "first_author": citation["first_author"],
        "journal": citation["journal"],
        "year": citation["year"],
    }


summary_rows = []
failed_pdbs = []

for pdb_id in pdb_ids:
    try:
        row = fetch_entry_summary(pdb_id)
        summary_rows.append(row)
        logger.info("Summary OK: %s", pdb_id)
    except Exception as exc:
        failed_pdbs.append(pdb_id)
        logger.error("Summary failed for %s: %s", pdb_id, exc)

summary_df = pd.DataFrame(summary_rows, columns=[
    "pdb_id", "title", "resolution_A", "doi", "first_author", "journal", "year"
])

if not summary_df.empty:
    summary_df["year"] = summary_df["year"].astype("Int64")

print(f"Fetched summary: {len(summary_df)} / {len(pdb_ids)}")
if failed_pdbs:
    print("Failed PDBs:", failed_pdbs)



In [ ]:
from __future__ import annotations

# Cell 4: 各 PDB の polymer entity から chain 情報を取得
# chain_id には auth_asym_id を使用します。
# 1 entity が複数 chain に対応する場合（例: B と D が同じ entity）は chain ごとに行を展開します。


def fetch_polymer_chains(pdb_id: str) -> list[dict]:
    """1 PDB の polymer entity から chain 行リストを返す。"""
    pdb_id = pdb_id.upper()
    entry_url = f"{RCSB_CORE_URL}/entry/{pdb_id}"
    entry = fetch_json(entry_url)
    entity_ids = entry.get("rcsb_entry_container_identifiers", {}).get("polymer_entity_ids", [])

    rows = []
    for entity_id in entity_ids:
        entity_url = f"{RCSB_CORE_URL}/polymer_entity/{pdb_id}/{entity_id}"
        entity = fetch_json(entity_url)
        description = entity.get("rcsb_polymer_entity", {}).get("pdbx_description", "Unknown")
        auth_asym_ids = entity.get("rcsb_polymer_entity_container_identifiers", {}).get("auth_asym_ids", [])

        # auth_asym_id が無い場合は pdbx_strand_id を fallback
        if not auth_asym_ids:
            strand_ids = entity.get("entity_poly", {}).get("pdbx_strand_id", "")
            auth_asym_ids = [s.strip() for s in strand_ids.replace(" ", "").split(",") if s.strip()]

        for chain_id in auth_asym_ids:
            rows.append({
                "pdb_id": pdb_id,
                "chain_id": chain_id,
                "entity_description": description,
            })
    return rows


chain_rows = []
chain_failed_pdbs = []

for pdb_id in pdb_ids:
    # summary 取得に失敗した PDB は chain もスキップ
    if pdb_id in failed_pdbs:
        continue
    try:
        rows = fetch_polymer_chains(pdb_id)
        chain_rows.extend(rows)
        logger.info("Chain OK: %s (%d chains)", pdb_id, len(rows))
    except Exception as exc:
        chain_failed_pdbs.append(pdb_id)
        logger.error("Chain fetch failed for %s: %s", pdb_id, exc)

chain_df = pd.DataFrame(chain_rows, columns=["pdb_id", "chain_id", "entity_description"])

# summary_chain_df: PDB ごとの chain 構成確認用（sorted long format）
# 例) 6XEZ / A / RNA-directed RNA polymerase
summary_chain_df = chain_df.sort_values(["pdb_id", "chain_id"]).reset_index(drop=True)

# PDB ごとの compact summary 文字列（確認用）
# 例) A: RNA-directed RNA polymerase; B: Non-structural protein 8; ...
def build_chain_summary(group: pd.DataFrame) -> str:
    parts = [f"{row.chain_id}: {row.entity_description}" for row in group.itertuples(index=False)]
    return "; ".join(parts)

chain_summary_text = (
    summary_chain_df.groupby("pdb_id", sort=True)
    .apply(build_chain_summary, include_groups=False)
    .reset_index(name="chain_summary")
)

print(f"Fetched chains: {len(chain_df)} rows from {summary_chain_df['pdb_id'].nunique()} PDBs")
if chain_failed_pdbs:
    print("Chain fetch failed:", chain_failed_pdbs)

print("\n--- Chain summary per PDB ---")
for _, row in chain_summary_text.iterrows():
    print(f"{row.pdb_id}: {row.chain_summary}")



In [ ]:
from __future__ import annotations

# Cell 5: 並べ替え & 表示
# chain_df を pdb_id, chain_id で sort してから表示します。

chain_df = chain_df.sort_values(["pdb_id", "chain_id"]).reset_index(drop=True)
summary_chain_df = chain_df.copy()

print("=== summary_df ===")
display(summary_df)

print("\n=== chain_df ===")
display(chain_df)

print("\n=== summary_chain_df ===")
display(summary_chain_df)



In [ ]:
from __future__ import annotations

# Cell 6: CSV 保存
# RSCC / BSA 解析前の確認用テーブルを CSV 出力します。

summary_csv = OUTPUT_DIR / "pdb_summary.csv"
chain_csv = OUTPUT_DIR / "pdb_chain_info.csv"
summary_df.to_csv(summary_csv, index=False)
chain_df.to_csv(chain_csv, index=False)

print(f"Saved: {summary_csv}")
print(f"Saved: {chain_csv}")

# 保存内容の先頭行を確認
print("\n--- pdb_summary.csv (head) ---")
print(summary_df.head().to_string(index=False))

print("\n--- pdb_chain_info.csv (head) ---")
print(chain_df.head(10).to_string(index=False))

